In [1]:
%load_ext autoreload
%autoreload 2


# Import Libraries

In [93]:
import os

import pandas as pd
import numpy as np
import nltk
import spacy
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from gensim.similarities.annoy import AnnoyIndexer

import gensim.models
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import random


nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/vaa2804/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Read Data

In [3]:
path = "../../"
train = pd.read_csv(os.path.join(path, "train.csv"))
test = pd.read_csv(os.path.join(path, "test.csv"))
print("Number of rows and columns in the train data set:", train.shape)
print("Number of rows and columns in the test data set:", test.shape)
train.head()

Number of rows and columns in the train data set: (48665, 2)
Number of rows and columns in the test data set: (12167, 2)


,rate,text
0,4,Очень понравилось. Были в начале марта с соба...
1,5,В целом магазин устраивает.\nАссортимент позво...
2,5,"Очень хорошо что открылась 5 ка, теперь не над..."
3,3,Пятёрочка громко объявила о том как она заботи...
4,3,"Тесно, вечная сутолока, между рядами трудно ра..."


In [4]:
train['rate'].value_counts()

rate
5    26069
4     9922
3     6126
1     4138
2     2410
Name: count, dtype: int64

# Label encoding

In [5]:
le = LabelEncoder()

train.rate = le.fit_transform(train.rate)
train.head()

,rate,text
0,3,Очень понравилось. Были в начале марта с соба...
1,4,В целом магазин устраивает.\nАссортимент позво...
2,4,"Очень хорошо что открылась 5 ка, теперь не над..."
3,2,Пятёрочка громко объявила о том как она заботи...
4,2,"Тесно, вечная сутолока, между рядами трудно ра..."


# Text Preprocessing

In [109]:
nlp = spacy.load("ru_core_news_sm")
stopwords = nlp.Defaults.stop_words
print(f'Spacy ru stopwords size: {len(stopwords)}', end='\n\n')
' '.join(stopwords)

Spacy ru stopwords size: 768



'нужные да ежели какой таки нашей теперь е насилу немного сначала того одному хотеть вроде одним своею нашу моей которою нипочем своей хоть никем таком необходимости чтоб этакий можете данного есть хочешь ничем всеми иными ними ешь некоторые будете спустя стала любыми рядом настоящие впрочем ю имела благодаря более ишь какого эй подобно наизнанку от сам либо немногим каким необходимо ко самая меня нее конечно её чему то никакой разве моими мою следует однажды ти алло будут менее ведь действительно увы которая затем отовсюду подобных мной ем одних едят даже другая многом у вдруг комья кой еле именно ела такими нужно скорее перед му вернее ваш наверху ею сей вся э слишком мне какими оне ничто пожалуйста ее него данное самых отчего доколе вновь внизу вресноту любого прочего аж саму весь бывает н своя эдакий нас щ негде ну насчет никогда сколько нынешней ща напротив чего щас какою данных вокруг любую она которого теми довольно ближайшие самими они своего проще ух чхать экий самих ниоткуда 

In [7]:
nltk_stopwords = nltk.corpus.stopwords.words("russian")
print(f'NLTK ru stopwords size: {len(nltk_stopwords)}', end='\n\n')
' '.join(nltk_stopwords)


NLTK ru stopwords size: 151



'и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее мне было вот от меня еще нет о из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между'

In [113]:
train['cleaned_text'] = train['text'].apply(
    lambda x: ' '.join(
        token.lemma_.lower() for token in nlp(x) if
        #not token.is_stop
        #not token.is_punct
        not token.is_digit
        and not token.like_email
        and not token.like_num
        and not token.is_space
    )
)

In [114]:
print(train['text'][:10])

0    Очень понравилось. Были в начале марта  с соба...
1    В целом магазин устраивает.\nАссортимент позво...
2    Очень хорошо что открылась 5 ка, теперь не над...
3    Пятёрочка громко объявила о том как она заботи...
4    Тесно, вечная сутолока, между рядами трудно ра...
5    Магазин в пешей доступности. После ремонта и р...
6    Магазин хороший цены и скидки нормальные токо ...
7    Редко сюда забегаю. Маленький магазинчик, но э...
8    Сложно найти в торговом центре. А магазин - норм)
9    После ремонта магазин в нутри стал ещё лучше. ...
Name: text, dtype: object


In [115]:
print(train['cleaned_text'][:10])

0    очень понравиться . были в начало март с собак...
1    в целое магазин устраивать . ассортимент позво...
2    очень хороший что открыться ка , теперь не над...
3    громко объявить о том как она заботиться о пен...
4    тесно , вечный сутолока , между ряд трудный ра...
5    магазин в пеший доступность . после ремонт и р...
6    магазин хороший цена и скидка нормальный токо ...
7    редко сюда забегать . маленький магазинчик , н...
8    сложный найти в торговый центр . а магазин - н...
9    после ремонт магазин в нутри стать ещё хороший...
Name: cleaned_text, dtype: object


In [116]:
train_1 = train[train['rate'] == 1]

In [152]:
train_2 = train[train['rate'] == 2]

In [117]:
import gensim.downloader as api
from gensim.models import KeyedVectors
from pymorphy3 import MorphAnalyzer
w2v_model = KeyedVectors.load_word2vec_format('~/gensim-data/ruwikiruscorpora-nobigrams_upos_skipgram_300_5_2018.vec.gz', binary=False,encoding='utf-8')


In [118]:
def get_synonyms(word, topn=5):
    try:
        return w2v_model.most_similar(word, topn=topn)
    except KeyError:
        return []

In [119]:
def get_syn (word) :
    morph = MorphAnalyzer()
    normal_form = morph.parse(word)
    syn  = normal_form[0][2] + '_' + str(normal_form[0][1]).split(',')[0]
    synonyms = get_synonyms(syn)
    if synonyms == [] :
        return None
    syn = synonyms[0][0].split('_')[0]
    return syn


In [120]:
def aug_sent(sentence, p = 0.8) :
    for i in range(len(sentence)) :
        r = random.random()
        if r < p :                   
            syn = get_syn(sentence[i])
            if syn != None :
                sentence[i] = syn
    return sentence

In [99]:
text = ['Очень', 'понравилось','.', 'Были', 'в', 'начале', 'марта', 'с', 'собакой']
print(aug_sent(text))

['Очень', 'нравиться', '.', 'Были', 'в', 'конец', 'февраль', 'с', 'пес']


In [145]:
X_train, X_test, y_train, y_test = train_test_split(train['cleaned_text'], train['rate'], shuffle = True, random_state=2025)

In [147]:
sent_list = []
rate_list = []
for index, row in train_1.iterrows():
    sent = row["cleaned_text"].split()
    new_sent = aug_sent(sent, p= 0.8)
    sent_str = ' '.join(new_sent)
    sent_list.append(sent_str)
    rate_list.append(1)
    

In [148]:
X_train = pd.concat([X_train, pd.Series(sent_list)], ignore_index=True)

In [150]:
y_train = pd.concat([y_train,pd.Series(rate_list)],ignore_index = True)

In [153]:
sent_list = []
rate_list = []
for index, row in train_2.iterrows():
    sent = row["cleaned_text"].split()
    new_sent = aug_sent(sent, p= 0.8)
    sent_str = ' '.join(new_sent)
    sent_list.append(sent_str)
    rate_list.append(2)
X_train = pd.concat([X_train, pd.Series(sent_list)], ignore_index=True)
y_train = pd.concat([y_train,pd.Series(rate_list)],ignore_index = True)

KeyboardInterrupt: 

In [151]:
model = make_pipeline(
    CountVectorizer(max_df=0.7),
    TruncatedSVD(n_components=500, n_iter=25, random_state=2023),
    LogisticRegression(max_iter = 200),
)
model.fit(X_train, y_train)
predicted_categories = model.predict(X_test)
print(classification_report(y_test, predicted_categories))

              precision    recall  f1-score   support

           0       0.58      0.49      0.53       998
           1       0.18      0.11      0.14       584
           2       0.42      0.32      0.36      1555
           3       0.49      0.28      0.36      2509
           4       0.72      0.92      0.81      6521

    accuracy                           0.64     12167
   macro avg       0.48      0.43      0.44     12167
weighted avg       0.60      0.64      0.60     12167



In [102]:
x_array = train['text'].to_numpy()
x_array=x_array.tolist()  

In [66]:
model = gensim.models.Word2Vec(
    sentences=x_array,
    vector_size=256, # default = 100
    window=7, # default = 5
    min_count=10,
    sg=1, # Training algorithm: 1 for skip-gram; otherwise CBOW
    hs=0, #  If 1, hierarchical softmax will be used for model training. If 0, and negative is non-zero, negative sampling will be used.
    negative=5, # If > 0, negative sampling will be used, if set to 0, no negative sampling is used.
    epochs=25, # Number of iterations (epochs) over the corpus
    seed=2023,
)

In [67]:
def text_to_vector(text):
    vectors = [model.wv[word] for word in text if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

In [68]:
X = np.array([text_to_vector(text) for text in x_array])


In [69]:
X_train, X_test, y_train, y_test = train_test_split(X, train['rate'], shuffle = True, random_state=2025)

In [72]:
logreg = LogisticRegression( random_state = 42,max_iter = 200)
logreg.fit(X_train, y_train)
preds = logreg.predict(X_test)
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.49      0.57      0.53       998
           1       0.09      0.01      0.01       584
           2       0.38      0.33      0.35      1555
           3       0.43      0.23      0.30      2509
           4       0.72      0.92      0.81      6521

    accuracy                           0.63     12167
   macro avg       0.42      0.41      0.40     12167
weighted avg       0.57      0.63      0.58     12167



In [74]:
tfidf = TfidfVectorizer(vocabulary=model.wv.key_to_index.keys())
tfidf.fit([" ".join(text) for text in x_array])
max_idf = max(tfidf.idf_)
word_weights = {word: tfidf.idf_[i] for word, i in tfidf.vocabulary_.items()}


In [75]:
def weighted_text_to_vector(text):
    vectors = []
    weights = []
    for word in text:
        if word in model.wv and word in word_weights:
            vectors.append(model.wv[word])
            weights.append(word_weights[word])
    
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    
    weights = np.array(weights)
    weights = weights / weights.sum()  # Нормализация весов
    return np.dot(weights, vectors)

In [76]:
X = np.array([weighted_text_to_vector(text) for text in x_array])

In [77]:
X_train, X_test, y_train, y_test = train_test_split(X, train['rate'], shuffle = True, random_state=2025)
logreg = LogisticRegression( random_state = 42,max_iter = 200)
logreg.fit(X_train, y_train)
preds = logreg.predict(X_test)
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.48      0.57      0.52       998
           1       0.20      0.02      0.04       584
           2       0.37      0.31      0.34      1555
           3       0.43      0.23      0.30      2509
           4       0.72      0.91      0.81      6521

    accuracy                           0.62     12167
   macro avg       0.44      0.41      0.40     12167
weighted avg       0.57      0.62      0.58     12167



In [79]:
tagged_data = [TaggedDocument(words=text, tags=[str(i)]) 
               for i, text in enumerate(x_array)]

In [83]:
model = Doc2Vec(
    documents=tagged_data,
    vector_size= 256,      # размерность векторов документа
    window=5,            # размер окна контекста
    min_count=1,         # минимальная частота слова
    workers=4,           # количество потоков
    epochs=20,           # количество эпох обучения
    dm=1                 # 1 = PV-DM, 0 = PV-DBOW
)

In [84]:
doc_vectors = np.array([model.dv[str(i)] for i in range(len(x_array))])
# 5. Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    doc_vectors, train['rate'], test_size=0.2, random_state=42)
logreg = LogisticRegression( random_state = 42,max_iter = 200)
logreg.fit(X_train, y_train)
preds = logreg.predict(X_test)
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.53      0.52      0.52       823
           1       0.00      0.00      0.00       530
           2       0.35      0.23      0.28      1246
           3       0.42      0.21      0.28      1949
           4       0.68      0.93      0.78      5185

    accuracy                           0.61      9733
   macro avg       0.39      0.38      0.37      9733
weighted avg       0.53      0.61      0.55      9733



In [85]:
model = gensim.models.Word2Vec(
    sentences=x_array,
    vector_size=256, # default = 100
    window=7, # default = 5
    min_count=10,
    sg=1, # Training algorithm: 1 for skip-gram; otherwise CBOW
    hs=0, #  If 1, hierarchical softmax will be used for model training. If 0, and negative is non-zero, negative sampling will be used.
    negative=5, # If > 0, negative sampling will be used, if set to 0, no negative sampling is used.
    epochs=25, # Number of iterations (epochs) over the corpus
    seed=2023,
)

In [14]:
pipe = Pipeline(
    steps=[
        ('tfidf', TfidfVectorizer()),
        ('clf', LogisticRegression(max_iter = 200))
    ]
).fit(X_train, y_train)
preds = pipe.predict(X_test)
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.56      0.55      0.55       998
           1       0.25      0.03      0.05       584
           2       0.40      0.33      0.36      1555
           3       0.45      0.28      0.34      2509
           4       0.72      0.91      0.81      6521

    accuracy                           0.64     12167
   macro avg       0.47      0.42      0.42     12167
weighted avg       0.59      0.64      0.60     12167



In [36]:
nlp.Defaults.stop_words = nltk_stopwords
print(f'Spacy ru stopwords size: {len(nlp.Defaults.stop_words)}', end='\n\n')
' '.join(nlp.Defaults.stop_words)

Spacy ru stopwords size: 151



'и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее мне было вот от меня еще нет о из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между'

In [37]:
train['nlp_cleaned_text'] = train['text'].apply(
    lambda x: ' '.join(
        token.lemma_.lower() for token in nlp(x) if
        not token.is_stop
        and not token.is_punct
        and not token.is_digit
        and not token.like_email
        and not token.like_num
        and not token.is_space
    )
)
X_nlp_train, X__nlp_test, y_train, y_test = train_test_split(train['nlp_cleaned_text'], train['rate'], shuffle = True, random_state=2025)


In [39]:
pipe = Pipeline(
    steps=[
        ('tfidf', TfidfVectorizer()),
        ('clf', LogisticRegression(max_iter = 200))
    ]
).fit(X_nlp_train, y_train)
preds = pipe.predict(X__nlp_test)
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           1       0.56      0.55      0.55       998
           2       0.25      0.03      0.05       584
           3       0.40      0.33      0.36      1555
           4       0.45      0.28      0.34      2509
           5       0.72      0.91      0.81      6521

    accuracy                           0.64     12167
   macro avg       0.47      0.42      0.42     12167
weighted avg       0.59      0.64      0.60     12167



In [11]:
nltk_stopwords = nltk.corpus.stopwords.words("russian")

# Init tf-idf
vect_word = TfidfVectorizer(
    max_features=100,
    lowercase=True,
    analyzer="word",
    stop_words=nltk_stopwords,
    ngram_range=(1, 3),
    dtype=np.float32
)

In [12]:
# Train tf-idf
Clean_train = vect_word.fit_transform(train["text"])
# Map tf-idf on test
Clean_test = vect_word.transform(test["text"])
y_train = train["rate"]

In [16]:
vect_word.get_feature_names_out()

array(['акции', 'ассортимент', 'большой', 'большой выбор', 'бывает',
       'бывают', 'быстро', 'вежливые', 'вежливый', 'вежливый персонал',
       'вообще', 'время', 'всё', 'выбор', 'выбор товаров', 'выпечка',
       'дома', 'домом', 'других', 'ещё', 'зале', 'касс', 'касса',
       'кассах', 'кассе', 'кассир', 'кассиры', 'кассы',
       'кассы самообслуживания', 'качество', 'кофе', 'купить', 'магазин',
       'магазина', 'магазине', 'магазинов', 'маленький', 'мало', 'найти',
       'нормальный', 'нравится', 'нужно', 'обслуживание', 'обычная',
       'овощи', 'одна', 'особенно', 'отличный', 'отличный магазин',
       'очень', 'очень удобно', 'очередей', 'очереди', 'очередь',
       'парковка', 'персонал', 'постоянно', 'приветливый', 'приятно',
       'продавцы', 'продуктов', 'продукты', 'просто', 'пятерочка',
       'пятёрочка', 'работает', 'работают', 'расположение', 'рекомендую',
       'ремонта', 'рядом', 'рядом домом', 'самообслуживания', 'свежие',
       'сети', 'скидки', 'сотрудн

# Init Model

In [14]:
# Init logreg model
logreg = LogisticRegression(
    C=2,
    random_state=42
)

# Train Model

In [8]:
# Train logreg
logreg.fit(Clean_train, y_train)

/home/pc/Desktop/nlp_huawei_new2_task/venv/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(C=2, random_state=42)

# Create Predicts

In [9]:
# Predict probabilities
preds_proba = logreg.predict_proba(X_test)
# Get classes
preds = np.argmax(preds_proba, axis=1)
# pred_labels = le.inverse_transform(preds)

# Create submission

In [10]:
sample_submission = pd.read_csv(os.path.join(path, "sample_submission.csv"))
sample_submission["rate"] = preds
sample_submission.rate = le.inverse_transform(sample_submission.rate)
sample_submission.head()

,rate
0,5
1,5
2,5
3,3
4,5


In [11]:
sample_submission.to_csv("submission.csv", index=False)